# AMT — Qwen3.5-4B + IGED Indonesian GEC

Notebook ini menjalankan eksperimen QLoRA end-to-end untuk koreksi ejaan, tata bahasa, diksi, dan struktur kalimat bahasa Indonesia menggunakan `syauqie/IGED`.

**Urutan penggunaan:** jalankan dengan `RUN_MODE = 'smoke'`, kemudian `pilot`, lalu `final`. Mode `smoke` tetap mengunduh model dan menjalankan dua langkah training agar masalah environment, tokenizer, quantization, LoRA, dan trainer terdeteksi lebih awal.

IGED bukan dataset proofreading legal. Adapter yang dihasilkan hanya boleh menjadi kandidat language repair di AMT. Protected spans, RAG, validator, diff, dan accept/reject manusia tetap wajib dipertahankan.

## Kaggle prerequisites

Aktifkan **Internet** dan **GPU**. Dataset dan model bersifat publik; token Hugging Face tidak diperlukan kecuali rate limit terjadi. Notebook membaca `HF_TOKEN` atau `HF_HUB_TOKEN` dari Kaggle Secrets tanpa mencetak nilainya.

Output ditulis ke `/kaggle/working/amt-qwen35-iged/<run-id>`. Adapter PEFT adalah output utama. Konversi ke MLX untuk AMT dilakukan setelah adapter melewati evaluasi offline.

In [ ]:
%pip install -q -U 'transformers>=5.2.0' 'datasets>=4.0.0' 'accelerate>=1.10.0' 'peft>=0.18.0' 'trl>=0.27.0' 'bitsandbytes>=0.46.0' 'safetensors>=0.5.0'
print('Dependencies installed. Restart the Kaggle session only if the runtime asks for it.')

In [ ]:
import gc
import hashlib
import inspect
import json
import math
import os
import random
import re
import statistics
import time
from collections import Counter
from pathlib import Path

import numpy as np

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

RUN_MODE = os.environ.get('AMT_RUN_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'pilot', 'final'}:
    raise ValueError("AMT_RUN_MODE must be 'smoke', 'pilot', or 'final'")

MODEL_ID = 'Qwen/Qwen3.5-4B'
DATASET_ID = 'syauqie/IGED'
SEED = 42
MAX_LENGTH = 768

if Path('/kaggle/working').exists():
    OUTPUT_ROOT = Path('/kaggle/working/amt-qwen35-iged')
else:
    OUTPUT_ROOT = Path.cwd() / 'amt-qwen35-iged'
RUN_ID = os.environ.get('AMT_RUN_ID', time.strftime('%Y%m%d-%H%M%S'))
RUN_ROOT = OUTPUT_ROOT / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HF_HUB_TOKEN') or None

SMOKE_TRAIN_ROWS = 256
SMOKE_VAL_ROWS = 64
SMOKE_TEST_ROWS = 64
PILOT_TRAIN_ROWS = 20_000
PILOT_VAL_ROWS = 1_000
PILOT_TEST_ROWS = 1_000
FINAL_TRAIN_ROWS = 300_000
FINAL_VAL_ROWS = 2_000
FINAL_TEST_ROWS = 2_000
PILOT_MAX_STEPS = 200
FINAL_MAX_STEPS = 8_000
EVAL_GENERATION_ROWS = 256

DEFAULT_FINAL_CONFIG = {
    'name': 'fallback-lr1e-4-r16',
    'learning_rate': 1e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'gradient_accumulation_steps': 16,
}
PILOT_CONFIGS = [
    {
        'name': 'lr5e-5-r16',
        'learning_rate': 5e-5,
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'gradient_accumulation_steps': 16,
    },
    {
        'name': 'lr1e-4-r16',
        'learning_rate': 1e-4,
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'gradient_accumulation_steps': 16,
    },
    {
        'name': 'lr1e-4-r32',
        'learning_rate': 1e-4,
        'lora_r': 32,
        'lora_alpha': 64,
        'lora_dropout': 0.05,
        'gradient_accumulation_steps': 16,
    },
]

def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=json_default) + '\n',
        encoding='utf-8',
    )

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

set_seed()
write_json(RUN_ROOT / 'run_settings.json', {
    'run_mode': RUN_MODE,
    'run_id': RUN_ID,
    'model_id': MODEL_ID,
    'dataset_id': DATASET_ID,
    'seed': SEED,
    'max_length': MAX_LENGTH,
})
print(f'RUN_MODE={RUN_MODE} | RUN_ID={RUN_ID}')
print(f'Output: {RUN_ROOT}')

## Pin model and dataset revisions

Revision SHA disimpan di manifest. Jika Hub metadata tidak dapat diakses, notebook tetap memberi peringatan dan memakai revision default; untuk hasil final, ulangi setelah Internet aktif.

In [ ]:
from huggingface_hub import HfApi

MODEL_REVISION = None
DATASET_REVISION = None
revision_error = None
try:
    hub_api = HfApi(token=HF_TOKEN)
    MODEL_REVISION = hub_api.model_info(MODEL_ID, revision='main').sha
    DATASET_REVISION = hub_api.dataset_info(DATASET_ID, revision='main').sha
except Exception as error:
    revision_error = repr(error)
    print('WARNING: could not pin Hub revisions:', revision_error)

def revision_kwargs(revision):
    return {'revision': revision} if revision else {}

revision_manifest = {
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'revision_lookup_error': revision_error,
}
write_json(RUN_ROOT / 'hub_manifest.json', revision_manifest)
print(json.dumps(revision_manifest, indent=2))

## Load, audit, clean, and split IGED

IGED pada Dataset Viewer memiliki pasangan `src`/`trg`, kategori error, dan metadata split. Kode di bawah tetap memiliki fallback split berbasis seed jika metadata split tidak tersedia. Cleaning hanya melakukan normalisasi whitespace, menghapus baris kosong, dan deduplikasi pasangan exact; tidak mengubah target koreksi.

In [ ]:
from datasets import Dataset, load_dataset

def normalize_text(value):
    if value is None:
        return ''
    return ' '.join(str(value).replace(chr(160), ' ').split())

def cap_dataset(dataset, cap, seed):
    cap = min(int(cap), len(dataset))
    if cap >= len(dataset):
        return dataset
    return dataset.shuffle(seed=seed).select(range(cap))

def deduplicate_dataset(dataset):
    seen = set()
    keep_indices = []
    sources = dataset['src_clean']
    targets = dataset['trg_clean']
    for index, (source, target) in enumerate(zip(sources, targets)):
        pair_hash = hashlib.sha1(
            (source + chr(31) + target).encode('utf-8')
        ).hexdigest()
        if pair_hash not in seen:
            seen.add(pair_hash)
            keep_indices.append(index)
    return dataset.select(keep_indices)

def clean_capped_dataset(dataset, cap, seed, label):
    selected = cap_dataset(dataset, cap, seed)
    def clean_row(row):
        source = normalize_text(row.get('src', ''))
        target = normalize_text(row.get('trg', ''))
        category = normalize_text(row.get('category', 'unknown')) or 'unknown'
        valid = bool(source and target and chr(0) not in source and chr(0) not in target)
        return {
            'src_clean': source,
            'trg_clean': target,
            'category': category,
            'valid': valid,
        }
    cleaned = selected.map(
        clean_row,
        remove_columns=selected.column_names,
        desc=f'Cleaning {label}',
    )
    cleaned = cleaned.filter(lambda row: row['valid'], desc=f'Filtering {label}')
    cleaned = cleaned.remove_columns('valid')
    before_dedup = len(cleaned)
    cleaned = deduplicate_dataset(cleaned)
    print(f'{label}: selected={len(selected):,} valid={before_dedup:,} unique={len(cleaned):,}')
    return cleaned

def split_dataset(raw):
    if 'split' in raw.column_names:
        labels = sorted({normalize_text(value).lower() for value in raw['split']})
        train_labels = {'train'}
        validation_labels = {'validation', 'valid', 'val', 'dev'}
        test_labels = {'test'}
        train = raw.filter(lambda row: normalize_text(row['split']).lower() in train_labels, desc='Selecting train')
        validation = raw.filter(lambda row: normalize_text(row['split']).lower() in validation_labels, desc='Selecting validation')
        test = raw.filter(lambda row: normalize_text(row['split']).lower() in test_labels, desc='Selecting test')
        if len(train) and len(validation) and len(test):
            print('Using IGED split column:', labels)
            return {'train': train, 'validation': validation, 'test': test, 'split_values': labels}
        print('Split column did not contain complete train/validation/test labels; using deterministic fallback:', labels)
    first = raw.train_test_split(test_size=0.10, seed=SEED)
    held_out = first['test'].train_test_split(test_size=0.50, seed=SEED)
    return {
        'train': first['train'],
        'validation': held_out['train'],
        'test': held_out['test'],
        'split_values': [],
    }

def dataset_audit(dataset):
    sources = dataset['src_clean']
    targets = dataset['trg_clean']
    source_lengths = [len(value.split()) for value in sources]
    target_lengths = [len(value.split()) for value in targets]
    no_op_count = sum(source == target for source, target in zip(sources, targets))
    def quantiles(values):
        if not values:
            return {}
        ordered = sorted(values)
        return {
            'min': ordered[0],
            'median': statistics.median(ordered),
            'p95': ordered[min(len(ordered) - 1, math.floor(len(ordered) * 0.95))],
            'max': ordered[-1],
        }
    return {
        'rows': len(dataset),
        'no_op_rows': no_op_count,
        'no_op_ratio': no_op_count / len(dataset) if dataset else 0.0,
        'categories': dict(Counter(dataset['category'])),
        'source_word_lengths': quantiles(source_lengths),
        'target_word_lengths': quantiles(target_lengths),
    }

raw = load_dataset(
    DATASET_ID,
    split='train',
    token=HF_TOKEN,
    **revision_kwargs(DATASET_REVISION),
)
if not {'src', 'trg'}.issubset(raw.column_names):
    raise ValueError(f'IGED schema changed. Expected src/trg, got {raw.column_names}')
print(raw)
raw_splits = split_dataset(raw)

if RUN_MODE == 'smoke':
    caps = {'train': SMOKE_TRAIN_ROWS, 'validation': SMOKE_VAL_ROWS, 'test': SMOKE_TEST_ROWS}
elif RUN_MODE == 'pilot':
    caps = {'train': PILOT_TRAIN_ROWS, 'validation': PILOT_VAL_ROWS, 'test': PILOT_TEST_ROWS}
else:
    caps = {'train': FINAL_TRAIN_ROWS, 'validation': FINAL_VAL_ROWS, 'test': FINAL_TEST_ROWS}

train_clean = clean_capped_dataset(raw_splits['train'], caps['train'], SEED, 'train')
validation_clean = clean_capped_dataset(raw_splits['validation'], caps['validation'], SEED + 1, 'validation')
test_clean = clean_capped_dataset(raw_splits['test'], caps['test'], SEED + 2, 'test')

if RUN_MODE == 'final':
    pilot_train_clean = cap_dataset(train_clean, PILOT_TRAIN_ROWS, SEED + 100)
    pilot_validation_clean = cap_dataset(validation_clean, PILOT_VAL_ROWS, SEED + 101)
else:
    pilot_train_clean = train_clean
    pilot_validation_clean = validation_clean

audits = {
    name: dataset_audit(dataset)
    for name, dataset in {
        'train': train_clean,
        'validation': validation_clean,
        'test': test_clean,
        'pilot_train': pilot_train_clean,
        'pilot_validation': pilot_validation_clean,
    }.items()
}
write_json(RUN_ROOT / 'dataset_audit.json', {
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'raw_columns': raw.column_names,
    'raw_rows': len(raw),
    'raw_split_values': raw_splits['split_values'],
    'audits': audits,
})
print(json.dumps(audits, ensure_ascii=False, indent=2))

## Build prompt-completion data

TRL akan menghitung loss pada completion. Prompt sengaja meminta output teks saja agar model belajar transformasi minimal, bukan penjelasan panjang.

In [ ]:
INSTRUCTION = (
    'Perbaiki ejaan, tata bahasa, tanda baca, diksi, dan struktur kalimat '
    'bahasa Indonesia berikut. Pertahankan makna. Jangan menambah atau '
    'menghapus fakta. Keluarkan hanya teks hasil perbaikan tanpa penjelasan.'
)

def build_prompt(source):
    return f'{INSTRUCTION}\n\nTEKS:\n{source}\n\nHASIL:\n'

def format_prompt_completion(dataset, label):
    def format_row(row):
        return {
            'prompt': build_prompt(row['src_clean']),
            'completion': row['trg_clean'],
        }
    formatted = dataset.map(
        format_row,
        remove_columns=dataset.column_names,
        desc=f'Formatting {label}',
    )
    if formatted.column_names != ['prompt', 'completion']:
        raise ValueError(f'Unexpected SFT columns: {formatted.column_names}')
    return formatted

train_sft = format_prompt_completion(train_clean, 'train')
validation_sft = format_prompt_completion(validation_clean, 'validation')
pilot_train_sft = format_prompt_completion(pilot_train_clean, 'pilot train')
pilot_validation_sft = format_prompt_completion(pilot_validation_clean, 'pilot validation')
print(train_sft[0])
print(f'SFT rows: train={len(train_sft):,}, validation={len(validation_sft):,}')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
    **revision_kwargs(MODEL_REVISION),
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

length_probe = train_sft.select(range(min(512, len(train_sft))))
lengths = [
    len(tokenizer(row['prompt'] + row['completion'], add_special_tokens=True, truncation=False)['input_ids'])
    for row in length_probe
]
length_audit = {
    'sample_rows': len(lengths),
    'max_length': MAX_LENGTH,
    'truncated_in_probe': sum(length > MAX_LENGTH for length in lengths),
    'min': min(lengths) if lengths else None,
    'median': statistics.median(lengths) if lengths else None,
    'p95': sorted(lengths)[min(len(lengths) - 1, math.floor(len(lengths) * 0.95))] if lengths else None,
    'max': max(lengths) if lengths else None,
}
write_json(RUN_ROOT / 'tokenizer_audit.json', length_audit)
print(length_audit)
print('Tokenizer:', tokenizer.__class__.__name__, '| vocab:', len(tokenizer))

## Model, QLoRA, and generation helpers

The notebook uses 4-bit NF4 quantization for the base model and discovers compatible projection names from the actual Qwen3.5 module tree. This avoids hard-coding a target list that may not match the hybrid attention implementation.

In [ ]:
import torch
import torch.nn as nn

from transformers import AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU is required for QLoRA training.')

BF16 = bool(torch.cuda.is_bf16_supported())
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
TF32 = torch.cuda.get_device_capability(0)[0] >= 8
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda, '| BF16:', BF16, '| TF32:', TF32)

model_config = AutoConfig.from_pretrained(
    MODEL_ID,
    **revision_kwargs(MODEL_REVISION),
    token=HF_TOKEN,
)
print('Model type:', model_config.model_type)
print('Architectures:', getattr(model_config, 'architectures', None))

def load_base_model():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    kwargs = {
        'quantization_config': quantization_config,
        'device_map': 'auto',
        'trust_remote_code': False,
        'token': HF_TOKEN,
        **revision_kwargs(MODEL_REVISION),
    }
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=COMPUTE_DTYPE, **kwargs
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=COMPUTE_DTYPE, **kwargs
        )
    model.config.use_cache = False
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    return model

TARGET_LEAF_NAMES = {
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'in_proj_qkv', 'in_proj_a', 'in_proj_b', 'out_proj',
    'gate_proj', 'up_proj', 'down_proj',
}

def discover_target_modules(model):
    import bitsandbytes as bnb
    linear_types = [nn.Linear]
    for name in ('Linear4bit', 'Linear8bitLt'):
        candidate = getattr(bnb.nn, name, None)
        if candidate is not None:
            linear_types.append(candidate)
    linear_types = tuple(linear_types)
    found = set()
    for module_name, module in model.named_modules():
        if not isinstance(module, linear_types):
            continue
        leaf = module_name.rsplit('.', 1)[-1]
        if leaf in TARGET_LEAF_NAMES:
            found.add(leaf)
    targets = sorted(found)
    if not targets:
        raise RuntimeError('Could not discover Qwen3.5 LoRA target modules.')
    return targets

def input_device(model):
    for parameter in model.parameters():
        if parameter.device.type != 'meta':
            return parameter.device
    return torch.device('cuda:0')

def clean_generation(text):
    text = str(text).strip()
    if '</think>' in text:
        text = text.rsplit('</think>', 1)[-1]
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    if 'HASIL:' in text:
        text = text.split('HASIL:', 1)[-1].strip()
    return text.strip().strip('`').strip()

def generate_prediction(model, source, max_new_tokens=256):
    device = input_device(model)
    encoded = tokenizer(
        build_prompt(source),
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    continuation = generated[0, encoded['input_ids'].shape[1]:]
    return clean_generation(tokenizer.decode(continuation, skip_special_tokens=True))

def protected_signature(text):
    pattern = r'https?://\S+|\b[A-Z]{2,}[A-Z0-9/-]*\b|\b\d+(?:[.,:/-]\d+)*\b'
    return sorted(set(re.findall(pattern, text)))

def metric_text(text):
    return normalize_text(text).lower()

def evaluate_generation(model, dataset, limit, output_path=None):
    sample = cap_dataset(dataset, limit, SEED + 500)
    rows = []
    exact = 0
    normalized_exact = 0
    protected_ok = 0
    similarities = []
    from difflib import SequenceMatcher
    for row in sample:
        source = row['src_clean']
        target = row['trg_clean']
        prediction = generate_prediction(model, source)
        exact += prediction == target
        normalized_exact += metric_text(prediction) == metric_text(target)
        similarities.append(SequenceMatcher(None, metric_text(prediction), metric_text(target)).ratio())
        source_protected = protected_signature(source)
        prediction_protected = protected_signature(prediction)
        protected_preserved = set(source_protected).issubset(set(prediction_protected))
        protected_ok += protected_preserved
        rows.append({
            'source': source,
            'reference': target,
            'prediction': prediction,
            'exact_match': prediction == target,
            'protected_source': source_protected,
            'protected_prediction': prediction_protected,
            'protected_preserved': protected_preserved,
        })
    report = {
        'rows': len(rows),
        'exact_match': exact / len(rows) if rows else 0.0,
        'normalized_exact_match': normalized_exact / len(rows) if rows else 0.0,
        'protected_token_preservation': protected_ok / len(rows) if rows else 0.0,
        'character_similarity': statistics.mean(similarities) if similarities else 0.0,
        'examples': rows[:20],
    }
    if output_path:
        write_json(output_path, {'report': report, 'predictions': rows})
    return report

def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

## Trainer utilities

The compatibility helpers account for the `eval_strategy`/`evaluation_strategy`, `max_length`/`max_seq_length`, and `processing_class`/`tokenizer` naming differences across recent Transformers/TRL releases.

In [ ]:
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

def make_lora_config(model, config):
    targets = discover_target_modules(model)
    lora_config = LoraConfig(
        r=config['lora_r'],
        lora_alpha=config['lora_alpha'],
        lora_dropout=config['lora_dropout'],
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=targets,
    )
    return lora_config, targets

def supported_kwargs(callable_object, values):
    parameters = inspect.signature(callable_object).parameters
    accepts_kwargs = any(
        parameter.kind == inspect.Parameter.VAR_KEYWORD
        for parameter in parameters.values()
    )
    if accepts_kwargs:
        return values
    return {key: value for key, value in values.items() if key in parameters}

def make_sft_args(config, output_dir, max_steps, eval_steps, save_steps):
    parameters = inspect.signature(SFTConfig).parameters
    values = {
        'output_dir': str(output_dir),
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': config['gradient_accumulation_steps'],
        'gradient_checkpointing': True,
        'gradient_checkpointing_kwargs': {'use_reentrant': False},
        'learning_rate': config['learning_rate'],
        'num_train_epochs': 1,
        'max_steps': max_steps,
        'warmup_ratio': 0.03,
        'lr_scheduler_type': 'cosine',
        'optim': 'paged_adamw_8bit',
        # Qwen3.5 exposes a functools.partial forward on some Transformers builds.
        # TRL's default chunked_nll patch expects __func__ and crashes there.
        'loss_type': 'nll',
        'weight_decay': 0.01,
        'max_grad_norm': 1.0,
        'logging_steps': 10,
        'eval_steps': eval_steps,
        'save_steps': save_steps,
        'save_total_limit': 2,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'eval_loss',
        'greater_is_better': False,
        'save_safetensors': True,
        'report_to': 'none',
        'seed': SEED,
        'data_seed': SEED,
        'bf16': BF16,
        'fp16': not BF16,
        'tf32': TF32,
        'dataloader_num_workers': 2,
        'remove_unused_columns': False,
        'ddp_find_unused_parameters': False,
        'packing': False,
        'completion_only_loss': True,
    }
    if 'eval_strategy' in parameters:
        values['eval_strategy'] = 'steps'
    elif 'evaluation_strategy' in parameters:
        values['evaluation_strategy'] = 'steps'
    if 'max_length' in parameters:
        values['max_length'] = MAX_LENGTH
    elif 'max_seq_length' in parameters:
        values['max_seq_length'] = MAX_LENGTH
    return SFTConfig(**supported_kwargs(SFTConfig, values))

def make_trainer(model, train_dataset, eval_dataset, args, lora_config):
    values = {
        'model': model,
        'args': args,
        'train_dataset': train_dataset,
        'eval_dataset': eval_dataset,
        'peft_config': lora_config,
    }
    trainer_parameters = inspect.signature(SFTTrainer).parameters
    if 'processing_class' in trainer_parameters:
        values['processing_class'] = tokenizer
    elif 'tokenizer' in trainer_parameters:
        values['tokenizer'] = tokenizer
    else:
        raise RuntimeError('This TRL version exposes neither processing_class nor tokenizer.')
    return SFTTrainer(**supported_kwargs(SFTTrainer, values))

def train_one(train_dataset, eval_dataset, config, run_name, max_steps, eval_steps, save_steps, resume=None):
    run_dir = RUN_ROOT / 'runs' / run_name
    adapter_dir = run_dir / 'adapter'
    run_dir.mkdir(parents=True, exist_ok=True)
    if resume is None and run_name == 'final' and run_dir.exists():
        resume = get_last_checkpoint(str(run_dir))
    print(f'\n=== {run_name} ===')
    print('rows:', len(train_dataset), 'train /', len(eval_dataset), 'eval')
    print('config:', config)
    print('resume:', resume)
    model = load_base_model()
    try:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    except TypeError:
        model = prepare_model_for_kbit_training(model)
    lora_config, target_modules = make_lora_config(model, config)
    print('LoRA targets:', target_modules)
    args = make_sft_args(config, run_dir, max_steps, eval_steps, save_steps)
    trainer = make_trainer(model, train_dataset, eval_dataset, args, lora_config)
    started = time.time()
    train_result = trainer.train(resume_from_checkpoint=resume)
    eval_metrics = trainer.evaluate()
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(run_dir / 'tokenizer'))
    trainer.save_state()
    elapsed = time.time() - started
    result = {
        'run_name': run_name,
        'config': config,
        'target_modules': target_modules,
        'train_rows': len(train_dataset),
        'eval_rows': len(eval_dataset),
        'elapsed_seconds': elapsed,
        'train_metrics': train_result.metrics,
        'eval_metrics': eval_metrics,
        'eval_loss': eval_metrics.get('eval_loss'),
        'adapter_dir': str(adapter_dir),
        'best_checkpoint': trainer.state.best_model_checkpoint,
    }
    write_json(run_dir / 'training_result.json', result)
    print('eval_loss:', result['eval_loss'], '| elapsed minutes:', round(elapsed / 60, 2))
    unload_model(model)
    return result

def load_adapter(adapter_dir):
    base_model = load_base_model()
    base_model.config.use_cache = True
    model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False)
    model.eval()
    return model

## Smoke test

Smoke test ini memeriksa baseline generation, forward/backward dua langkah, checkpoint adapter, reload adapter, dan generation setelah training. Jika cell ini gagal, jangan lanjut ke pilot.

In [ ]:
base_model = load_base_model()
base_model.config.use_cache = True
baseline_report = evaluate_generation(
    base_model,
    test_clean,
    limit=min(8, len(test_clean)),
    output_path=RUN_ROOT / 'baseline_generation.json',
)
print('Baseline report:', baseline_report)
unload_model(base_model)

smoke_config = dict(DEFAULT_FINAL_CONFIG)
smoke_config['name'] = 'smoke'
smoke_train = pilot_train_sft.select(range(min(SMOKE_TRAIN_ROWS, len(pilot_train_sft))))
smoke_validation = pilot_validation_sft.select(range(min(SMOKE_VAL_ROWS, len(pilot_validation_sft))))
smoke_result = train_one(
    smoke_train,
    smoke_validation,
    smoke_config,
    run_name='smoke',
    max_steps=2,
    eval_steps=1,
    save_steps=1,
)

smoke_model = load_adapter(smoke_result['adapter_dir'])
smoke_report = evaluate_generation(
    smoke_model,
    test_clean,
    limit=min(8, len(test_clean)),
    output_path=RUN_ROOT / 'smoke_generation.json',
)
print('Smoke report:', smoke_report)
unload_model(smoke_model)
write_json(RUN_ROOT / 'smoke_summary.json', {
    'baseline': baseline_report,
    'training': smoke_result,
    'after_training': smoke_report,
})
print('Smoke test completed. Proceed only if the adapter reload and generation succeeded.')

## Pilot hyperparameter sweep

Pilot menjalankan tiga konfigurasi pada subset 20K dengan 200 optimizer steps. Pemilihan awal berdasarkan `eval_loss`; hasil tetap harus dilihat bersama contoh generation dan protected-token preservation.

In [ ]:
pilot_results = []
selected_config = dict(DEFAULT_FINAL_CONFIG)

if RUN_MODE in {'pilot', 'final'}:
    for pilot_config in PILOT_CONFIGS:
        result = train_one(
            pilot_train_sft,
            pilot_validation_sft,
            pilot_config,
            run_name=f'pilot-{pilot_config["name"]}',
            max_steps=PILOT_MAX_STEPS,
            eval_steps=50,
            save_steps=50,
        )
        pilot_results.append(result)
    valid_results = [
        result for result in pilot_results
        if result.get('eval_loss') is not None and math.isfinite(float(result['eval_loss']))
    ]
    if not valid_results:
        raise RuntimeError('All pilot configurations failed to produce a finite eval_loss.')
    best_result = min(valid_results, key=lambda result: float(result['eval_loss']))
    selected_config = dict(best_result['config'])
    print('Selected pilot configuration:', selected_config)
else:
    print('RUN_MODE=smoke: pilot sweep skipped.')

write_json(RUN_ROOT / 'pilot_results.json', {
    'results': pilot_results,
    'selected_config': selected_config,
})

## Final training

Mode `final` menggunakan row cap 300K, sequence length 768, effective batch size 16, dan maksimum 8.000 optimizer steps. Checkpoint terakhir pada folder run yang sama akan dipakai otomatis; untuk sesi baru, set `AMT_RUN_ID` yang sama dan/atau `AMT_RESUME_CHECKPOINT` ke path checkpoint.

In [ ]:
final_result = None
final_adapter_dir = None

if RUN_MODE == 'final':
    final_config = dict(selected_config)
    final_config['name'] = 'final'
    resume_checkpoint = os.environ.get('AMT_RESUME_CHECKPOINT') or None
    final_result = train_one(
        train_sft,
        validation_sft,
        final_config,
        run_name='final',
        max_steps=FINAL_MAX_STEPS,
        eval_steps=250,
        save_steps=250,
        resume=resume_checkpoint,
    )
    final_adapter_dir = final_result['adapter_dir']
    write_json(RUN_ROOT / 'final_config.json', final_config)
    print('Final adapter:', final_adapter_dir)
else:
    print('RUN_MODE is not final: final training skipped.')

## Generation evaluation and artifact manifest

Generation metrics di sini bersifat diagnostic, bukan bukti ketepatan hukum. Periksa contoh hasil, terutama perubahan angka, identifier, negasi, dan kalimat yang seharusnya tidak diubah.

In [ ]:
evaluation_result = None
evaluation_adapter = None
if RUN_MODE == 'final':
    evaluation_adapter = final_adapter_dir
elif RUN_MODE == 'pilot' and pilot_results:
    best_pilot = min(
        [result for result in pilot_results if result.get('eval_loss') is not None],
        key=lambda result: float(result['eval_loss']),
    )
    evaluation_adapter = best_pilot['adapter_dir']

if evaluation_adapter:
    evaluation_model = load_adapter(evaluation_adapter)
    tuned_report = evaluate_generation(
        evaluation_model,
        test_clean,
        limit=min(EVAL_GENERATION_ROWS, len(test_clean)),
        output_path=RUN_ROOT / 'tuned_generation.json',
    )
    evaluation_result = {
        'baseline': baseline_report,
        'tuned': tuned_report,
        'adapter': evaluation_adapter,
    }
    write_json(RUN_ROOT / 'generation_comparison.json', evaluation_result)
    print(json.dumps(evaluation_result, ensure_ascii=False, indent=2))
    unload_model(evaluation_model)
else:
    print('No pilot/final adapter selected for generation evaluation.')

artifact_manifest = {
    'run_id': RUN_ID,
    'run_mode': RUN_MODE,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'max_length': MAX_LENGTH,
    'final_max_steps': FINAL_MAX_STEPS,
    'selected_config': selected_config,
    'baseline_report': baseline_report,
    'evaluation_result': evaluation_result,
    'legal_use_warning': 'IGED is general Indonesian GEC data, not lawyer-reviewed legal proofreading data.',
}
write_json(RUN_ROOT / 'artifact_manifest.json', artifact_manifest)

for path in sorted(RUN_ROOT.rglob('*')):
    if path.is_file():
        print(path.relative_to(RUN_ROOT), path.stat().st_size, 'bytes')
print('\nRun artifacts are ready under:', RUN_ROOT)

## Interpretation checklist

- Jangan memilih konfigurasi hanya dari loss; baca `pilot-*/training_result.json` dan contoh `tuned_generation.json`.
- Evaluasi legal AMT harus menggunakan dokumen holdout yang tidak berasal dari IGED dan ditinjau manusia.
- Model tidak boleh menerapkan rewrite otomatis. Gunakan adapter sebagai candidate proposer di balik parser, validator, protected-span checks, diff, dan accept/reject.
- Output notebook adalah adapter Transformers/PEFT. Jangan menyalin adapter langsung ke bundle MLX tanpa conversion dan smoke test runtime yang terpisah.